# 04_M4_Physics_Loss_Only.ipynb

M4: generic 19-feature vibration input with the shared signal-derived physics-consistency loss.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/pinn-motor-fault-code-v2')]
PROJECT_ROOT = next((path.resolve() for path in ROOT_CANDIDATES if (path / 'src' / 'pinn_motor_fault').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Cannot locate the repository source tree. Set the notebook working directory to the project root.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
REPORTS_DIR = PROJECT_ROOT / 'reports'
if os.environ.get('PINN_PADERBORN_DATA'):
    DATA_DIR = Path(os.environ['PINN_PADERBORN_DATA']).expanduser().resolve()
else:
    KAGGLE_DATA_DIR = Path('/kaggle/input/paderborn')
    LOCAL_DATA_DIR = PROJECT_ROOT / 'data' / 'paderborn' / 'extracted'
    DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.is_dir() else LOCAL_DATA_DIR
print('Project root:', PROJECT_ROOT)
print('Paderborn data:', DATA_DIR)


## Reproducible configuration
The full experiment runs only on real Paderborn recordings. The exact recording population and shared split are required; missing inputs and loader errors are not caught or replaced with synthetic data.

## Shared physics-target definition
For every raw vibration window, independently of classifier feature group, the target partitions 5-2000 Hz envelope-spectrum power. Healthy is the fraction outside the BPFO/BPFI/BSF harmonic bands. Inner-race, outer-race, and rolling-element components are the fractions assigned to the nearest corresponding harmonic within one native FFT-bin tolerance. The four nonnegative energies are normalized to sum to one. This is a label-independent spectral-consistency target, not a ground-truth label or calibrated posterior. M4 and M6 call the same target function on the same raw vibration windows. Binary experiments map it to `[p_healthy, p_inner + p_outer + p_rolling]`.

In [ ]:
from pinn_motor_fault.paderborn import MODALITY_SAMPLE_RATES_HZ, load_paderborn_windows_with_metadata
from pinn_motor_fault.train import run_experiment

EXPERIMENT_ID = 'M4_four_generic_physics'
TASK = 'four'
FEATURE_GROUP = 'generic'
USE_PHYSICS_LOSS = True
WINDOW_SIZE = 4096
STRIDE = 2048
MAX_WINDOWS_PER_RECORDING = 16
SEED = 17
HIDDEN_DIM = 48
LEARNING_RATE = 0.015
BATCH_SIZE = 64
EPOCHS = 60
assert DATA_DIR.is_dir(), f'Paderborn data directory does not exist: {DATA_DIR}'
loaded = load_paderborn_windows_with_metadata(
    DATA_DIR, window_size=WINDOW_SIZE, stride=STRIDE,
    signal_preference='vibration', max_windows_per_file=MAX_WINDOWS_PER_RECORDING,
    require_modality_match=True,
)
assert loaded.modality == 'vibration'
assert loaded.windows.size > 0 and len(loaded.sources) == loaded.labels.size
assert all(Path(source).suffix.lower() == '.mat' for source in loaded.sources), 'Expected real Paderborn MAT recordings.'
data_root = DATA_DIR.resolve()
assert all(Path(source).resolve().is_relative_to(data_root) for source in loaded.sources), 'Loaded recordings must be under DATA_DIR.'
split_manifest = REPORTS_DIR / 'shared' / 'split_manifest.json'
assert split_manifest.is_file(), 'Run 00_Project_Setup_and_Data_Audit.ipynb first to create the shared real-recording split.'
experiment_dir = REPORTS_DIR / EXPERIMENT_ID
metrics = run_experiment(
    loaded.windows, loaded.labels, loaded.sources, experiment_dir,
    task=TASK, feature_group=FEATURE_GROUP,
    use_physics_loss=USE_PHYSICS_LOSS, physics_weight=0.20 if USE_PHYSICS_LOSS else 0.0,
    hidden_dim=HIDDEN_DIM, learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE, epochs=EPOCHS, seed=SEED,
    split_manifest=split_manifest, experiment_id=EXPERIMENT_ID,
    dataset_name='real_paderborn_vibration',
)
print(json.dumps({key: metrics[key] for key in ('accuracy', 'macro_f1', 'balanced_accuracy')}, indent=2))
print('Artifacts:', experiment_dir.resolve())


## Output artifacts
Each fixed experiment ID writes its own model checkpoint, configuration, exact split, metrics, predictions, CSV summaries, and figures. The training code refuses to overwrite a non-empty experiment directory.